# 📊 App TrendPulse — Optimized Analytics

**Self-contained Jupyter Notebook** — all code runs inline, no external modules needed.

| Section | Description |
|---------|------------|
| 1. Setup & Imports | Install deps + import libraries |
| 2. Configuration | App list + constants |
| 3. Data Collection | Fetch app metadata + reviews with error handling |
| 4. Feature Engineering | Clean data, derive monetization, compute scores |
| 5. Pipeline Execution | Run the full pipeline (data fetch → features) |
| 6. Machine Learning | Cross-validated RandomForest |
| 7. Clustering | KMeans with dynamic label assignment |
| 8. Recommendation Engine | Scaled cosine similarity |
| 9. Visualizations | Plotly interactive charts |
| 10. Trend Analysis | Rolling average (replaces invalid LSTM) |
| 11. Export | Save dataset + model |

## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment on first run)
# !pip install google-play-scraper vaderSentiment plotly scikit-learn joblib

In [1]:
import os
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import joblib
import plotly.express as px
import plotly.graph_objects as go

from google_play_scraper import app as gplay_app, reviews as gplay_reviews
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

# Suppress noisy warnings
warnings.filterwarnings("ignore", category=UserWarning)
os.environ["OMP_NUM_THREADS"] = "1"

RANDOM_STATE = 42
print("✅ All imports loaded successfully")

✅ All imports loaded successfully


## 2. Configuration

In [2]:
APPS_LIST = [
    "com.spotify.music",
    "com.instagram.android",
    "com.duolingo",
    "com.whatsapp",
    "com.netflix.mediaclient",
    "com.facebook.katana",
    "com.snapchat.android",
    "com.google.android.youtube",
    "com.amazon.mShop.android.shopping",
    "com.twitter.android",
    "com.microsoft.office.word",
    "com.adobe.lrmobile",
    "com.canva.editor",
    "com.fitbit.FitbitMobile",
    "com.strava",
]

REVIEW_COUNT = 200          # reviews to scrape per app
OUTPUT_DIR   = "../app"     # where to save dataset.csv and model.pkl

print(f"Tracking {len(APPS_LIST)} apps, {REVIEW_COUNT} reviews each")

Tracking 15 apps, 200 reviews each


## 3. Data Collection Functions

Key improvements over original:
- **Error handling** with try/except so one failed app doesn't crash everything
- **Real monetization fields** (`free`, `containsAds`, `offersIAP`) from Google Play
- **Progress logging** so you can see which apps are being scraped

In [3]:
def fetch_app_data(app_ids):
    """Scrape app metadata from Google Play Store.
    
    Returns DataFrame with: App, Rating, Installs, Reviews, Genre,
    Free (bool), ContainsAds (bool), IAP (bool).
    """
    data = []
    for app_id in app_ids:
        try:
            result = gplay_app(app_id)
            data.append({
                "App":         result["title"],
                "App_ID":      app_id,
                "Rating":      result["score"],
                "Installs":    result["installs"],
                "Reviews":     result["ratings"],
                "Genre":       result["genre"],
                # Real monetization data (not random!)
                "Free":        result.get("free", True),
                "ContainsAds": result.get("containsAds", False),
                "IAP":         result.get("offersIAP", False),
            })
            print(f"  ✔ {result['title']}")
        except Exception as e:
            print(f"  ✖ Skipped {app_id} — {e}")
    
    return pd.DataFrame(data)

print("✅ fetch_app_data defined")

✅ fetch_app_data defined


In [ ]:
def fetch_reviews_data(app_ids, count=REVIEW_COUNT):
    """Scrape user reviews with timestamps + VADER sentiment."""
    analyzer = SentimentIntensityAnalyzer()
    reviews_data = []

    for app_id in app_ids:
        try:
            app_info = gplay_app(app_id)
            app_name = app_info["title"]
            result, _ = gplay_reviews(app_id, count=count)

            for r in result:
                review_time = r.get("at")
                if review_time is None:
                    review_time = datetime.now() - timedelta(
                        days=np.random.randint(0, 30)
                    )

                reviews_data.append({
                    "App":         app_name,
                    "Review":      r.get("content", ""),
                    "Sentiment":   analyzer.polarity_scores(r["content"])["compound"],
                    "Review_Time": review_time,
                    "Score":       r.get("score", 0),
                })

            print(f"  ✔ {len(result)} reviews for {app_name}")
        except Exception as e:
            print(f"  ✖ Skipped reviews for {app_id} — {e}")

    df = pd.DataFrame(reviews_data)
    df["Review_Time"] = pd.to_datetime(df["Review_Time"], errors="coerce")
    return df

print("✅ fetch_reviews_data defined")

## 4. Feature Engineering Functions

Key fixes:
- **`derive_monetization()`** uses real Google Play fields instead of `np.random.choice`
- **`Engagement_Score`** formula unchanged but now built on real data
- **`Market_Share`** computed during processing, not in a separate cell

In [4]:
def derive_monetization(row):
    """Derive monetization label from real Google Play fields."""
    if not row.get("Free", True):
        return "Paid"
    if row.get("IAP", False):
        return "Freemium"
    if row.get("ContainsAds", False):
        return "Ad-Supported"
    return "Free"


def process_data(app_df, review_df):
    """Merge app metadata with review sentiment + engineer all features."""
    
    # Merge average sentiment per app
    sentiment_summary = (
        review_df.groupby("App")["Sentiment"]
        .mean()
        .reset_index()
    )
    df = pd.merge(app_df, sentiment_summary, on="App", how="left")

    # Clean installs column
    df["Installs"] = (
        df["Installs"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .astype(float)
    )
    df["Sentiment"] = df["Sentiment"].fillna(0)

    # Real monetization (not random!)
    df["Monetization"] = df.apply(derive_monetization, axis=1)

    # Engagement composite score
    df["Engagement_Score"] = (
        df["Rating"] * 0.5
        + df["Sentiment"] * 2
        + np.log10(df["Installs"] + 1) * 0.5
    )

    # Market share
    df["Market_Share"] = (df["Installs"] / df["Installs"].sum()) * 100

    return df

print("✅ process_data defined")

✅ process_data defined


In [ ]:
def add_time_features(df):
    """Extract temporal features from review timestamps.
    Returns a copy to avoid mutating the input DataFrame.
    """
    df = df.copy()

    if "Review_Time" not in df.columns:
        print("⚠️ Review_Time missing — generating fallback timestamps")
        df["Review_Time"] = pd.Timestamp.now()

    df["Review_Time"] = pd.to_datetime(df["Review_Time"], errors="coerce")
    df["Hour"] = df["Review_Time"].dt.hour
    df["Day"]  = df["Review_Time"].dt.day_name()
    df["Date"] = df["Review_Time"].dt.date

    return df

print("✅ add_time_features defined")

## 5. Run the Pipeline

> **Important:** This is a single pipeline call. The original notebook ran
> `fetch_app_data()` and `fetch_reviews_data()` **twice** in separate cells,
> making ~6,000 redundant API calls. This is now fixed.

In [5]:
print("=" * 50)
print("Step 1/3: Fetching app metadata...")
print("=" * 50)
apps_df = fetch_app_data(APPS_LIST)
print(f"\n✅ Fetched {len(apps_df)} apps")

print("\n" + "=" * 50)
print("Step 2/3: Fetching reviews...")
print("=" * 50)
reviews_df = fetch_reviews_data(APPS_LIST)
print(f"\n✅ Fetched {len(reviews_df)} reviews")

print("\n" + "=" * 50)
print("Step 3/3: Processing & feature engineering...")
print("=" * 50)
final_df = process_data(apps_df, reviews_df)
time_df = add_time_features(reviews_df)

print(f"✅ final_df: {final_df.shape}")
print(f"✅ time_df:  {time_df.shape}")

Step 1/3: Fetching app metadata...
  ✔ Spotify: Music and Podcasts
  ✔ Instagram
  ✔ Duolingo: Language & Chess
  ✔ WhatsApp Messenger
  ✔ Netflix
  ✔ Facebook
  ✔ Snapchat
  ✔ YouTube
  ✔ Amazon Shopping
  ✔ X
  ✔ Microsoft Word: Edit Documents
  ✔ Lightroom Photo & Video Editor
  ✔ Canva: AI Photo & Video Editor
  ✔ Fitbit
  ✔ Strava: Run, Bike, Walk

✅ Fetched 15 apps

Step 2/3: Fetching reviews...


NameError: name 'fetch_reviews_data' is not defined

In [6]:
# Preview dataset
final_df.head(15)

NameError: name 'final_df' is not defined

## 6. Machine Learning Model

Improvements:
- **`random_state=42`** for reproducibility (was missing)
- **`max_depth=3`** to prevent overfitting on 15 samples
- **Cross-validation** instead of single train/test split (unreliable with n=15)
- **Feature importance** analysis added

In [7]:
# Train RandomForest with cross-validation
X = final_df[["Installs", "Sentiment"]]
y = final_df["Rating"]

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,              # prevent overfitting on 15 rows
    random_state=RANDOM_STATE,
)

# Cross-validated R² (much more honest than single split with n=15)
cv_folds = min(5, len(final_df))
cv_scores = cross_val_score(model, X, y, cv=cv_folds, scoring="r2")

print(f"Cross-Validation ({cv_folds}-fold):")
for i, score in enumerate(cv_scores):
    print(f"  Fold {i+1}: R² = {score:.3f}")

print(f"\nMean R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Fit on full data for deployment
model.fit(X, y)
print("\n✅ Model trained on full dataset")

NameError: name 'final_df' is not defined

In [8]:
# Feature importance
feat_imp = pd.DataFrame({
    "Feature": ["Installs", "Sentiment"],
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=True)

fig = px.bar(
    feat_imp,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Feature Importance (RandomForest)",
    color="Importance",
    color_continuous_scale="Teal",
)
fig.update_layout(template="plotly_dark")
fig.show()

NameError: name 'model' is not defined

## 7. App Segmentation (KMeans Clustering)

**Fix:** The original hardcoded cluster labels (`{0: 'Low Engagement', ...}`).
Since KMeans cluster IDs are arbitrary, this could assign wrong labels.
Now we inspect centroid engagement scores to assign labels correctly.

In [9]:
# Cluster apps with dynamic label assignment
features = ["Rating", "Installs", "Sentiment", "Engagement_Score"]

cluster_scaler = StandardScaler()
scaled_data = cluster_scaler.fit_transform(final_df[features])

kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
final_df["Cluster"] = kmeans.fit_predict(scaled_data)

# Assign labels DYNAMICALLY by inspecting centroid engagement scores
centroids = pd.DataFrame(
    cluster_scaler.inverse_transform(kmeans.cluster_centers_),
    columns=features,
)
ranked = centroids["Engagement_Score"].rank().astype(int)
label_map = {1: "Low Engagement", 2: "Growth Apps", 3: "Top Performers"}
cluster_names = {i: label_map[ranked.iloc[i]] for i in range(3)}

final_df["Segment"] = final_df["Cluster"].map(cluster_names)

# Engagement level bins
final_df["Engagement_Level"] = pd.cut(
    final_df["Engagement_Score"],
    bins=3,
    labels=["Low", "Medium", "High"],
)

print("Segment distribution:")
print(final_df["Segment"].value_counts())

NameError: name 'final_df' is not defined

In [10]:
# Segmentation scatter plot
fig = px.scatter(
    final_df,
    x="Installs",
    y="Engagement_Score",
    color="Segment",
    size="Rating",
    hover_name="App",
    log_x=True,
    title="📊 App Segmentation: Engagement vs Installs",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(
    template="plotly_dark",
    xaxis_title="Total Installs (Log Scale)",
    yaxis_title="Engagement Score",
)
fig.show()

NameError: name 'final_df' is not defined

In [11]:
# Segment distribution pie
segment_pct = (
    final_df["Segment"].value_counts(normalize=True) * 100
).reset_index()
segment_pct.columns = ["Segment", "Percentage"]

fig = px.pie(
    segment_pct,
    names="Segment",
    values="Percentage",
    title="🤖 App Segmentation Distribution (%)",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_layout(template="plotly_dark")
fig.show()

NameError: name 'final_df' is not defined

## 8. Recommendation Engine

**Fix:** The original used raw (unscaled) features in cosine similarity.
Since Installs are in the billions while Sentiment is -1 to 1,
recommendations were dominated by install count alone.
Now we scale features first.

In [12]:
def build_similarity_matrix(df):
    """Compute cosine similarity on SCALED features."""
    feats = df[["Rating", "Sentiment", "Engagement_Score"]].fillna(0)
    sim_scaler = StandardScaler()
    scaled = sim_scaler.fit_transform(feats)
    return cosine_similarity(scaled)


def recommend(app_name, df, top_n=5):
    """Find the top-N most similar apps."""
    sim = build_similarity_matrix(df)
    matches = df[df["App"] == app_name]
    if matches.empty:
        print(f"⚠️ App '{app_name}' not found")
        return []
    
    idx = matches.index[0]
    scores = sorted(enumerate(sim[idx]), key=lambda x: x[1], reverse=True)
    return [df.iloc[i]["App"] for i, _ in scores[1 : top_n + 1]]


# Demo recommendations
print("Recommendations:")
print("-" * 60)
for app_name in final_df["App"].head(5):
    recs = recommend(app_name, final_df)
    print(f"  {app_name}")
    print(f"    → {recs}")
    print()

Recommendations:
------------------------------------------------------------


NameError: name 'final_df' is not defined

## 9. Visualizations

### 9a. Monetization Distribution

> Labels now come from **real** Google Play data, not `np.random.choice()`.

In [13]:
monet_pct = (
    final_df["Monetization"].value_counts(normalize=True) * 100
).reset_index()
monet_pct.columns = ["Strategy", "Percentage"]

fig = px.pie(
    monet_pct,
    names="Strategy",
    values="Percentage",
    title="💰 Monetization Strategy Distribution (%)",
    color_discrete_sequence=px.colors.sequential.Teal,
)
fig.update_layout(template="plotly_dark")
fig.show()

NameError: name 'final_df' is not defined

### 9b. Engagement Distribution

In [14]:
eng_pct = (
    final_df["Engagement_Level"].value_counts(normalize=True) * 100
).reset_index()
eng_pct.columns = ["Level", "Percentage"]

fig = px.pie(
    eng_pct,
    names="Level",
    values="Percentage",
    title="🔥 Engagement Level Distribution (%)",
    hole=0.4,
    color_discrete_sequence=px.colors.sequential.Mint,
)
fig.update_layout(template="plotly_dark")
fig.show()

NameError: name 'final_df' is not defined

### 9c. Market Share

In [15]:
top_apps = final_df.sort_values("Market_Share", ascending=False).head(10)

fig = px.bar(
    top_apps,
    x="App",
    y="Market_Share",
    title="🏆 Top 10 Apps by Market Share (%)",
    color="Market_Share",
    color_continuous_scale="Viridis",
    text=top_apps["Market_Share"].round(1),
)
fig.update_layout(
    template="plotly_dark",
    xaxis_title="",
    yaxis_title="Market Share (%)",
    xaxis_tickangle=-30,
)
fig.update_traces(textposition="outside")
fig.show()

NameError: name 'final_df' is not defined

In [16]:
# Genre market contribution
genre_share = (
    final_df.groupby("Genre")["Installs"]
    .sum()
    .reset_index()
)
genre_share["Percentage"] = (
    genre_share["Installs"] / genre_share["Installs"].sum()
) * 100

fig = px.bar(
    genre_share.sort_values("Percentage", ascending=True),
    x="Percentage",
    y="Genre",
    orientation="h",
    title="📊 Genre Market Contribution (%)",
    color="Percentage",
    color_continuous_scale="Teal",
)
fig.update_layout(template="plotly_dark")
fig.show()

NameError: name 'final_df' is not defined

### 9d. Temporal Analysis — Activity Heatmap

In [17]:
# Activity heatmap: Day vs Hour
heatmap_data = (
    time_df.groupby(["Day", "Hour"])
    .size()
    .reset_index(name="Activity")
)

pivot = heatmap_data.pivot(
    index="Day", columns="Hour", values="Activity"
).fillna(0)

# Order days properly
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = pivot.reindex([d for d in day_order if d in pivot.index])

fig = px.imshow(
    pivot,
    title="🔥 User Activity Heatmap (Hour vs Day)",
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"x": "Hour of Day", "y": "Day of Week", "color": "Activity"},
)
fig.update_layout(template="plotly_dark")
fig.show()

NameError: name 'time_df' is not defined

### 9e. Hourly & Daily Activity

In [18]:
# Hourly distribution
hourly = time_df.groupby("Hour").size().reset_index(name="Activity")
hourly["Percentage"] = (hourly["Activity"] / hourly["Activity"].sum()) * 100

fig = px.area(
    hourly,
    x="Hour",
    y="Percentage",
    title="⏰ Activity Distribution by Hour (%)",
    color_discrete_sequence=["#14b8a6"],
)
fig.update_layout(template="plotly_dark", xaxis=dict(dtick=1))
fig.show()

NameError: name 'time_df' is not defined

In [19]:
# Daily distribution
daily_act = time_df.groupby("Day").size().reset_index(name="Activity")
daily_act["Percentage"] = (daily_act["Activity"] / daily_act["Activity"].sum()) * 100

fig = px.bar(
    daily_act,
    x="Day",
    y="Percentage",
    title="📅 Activity by Day of Week (%)",
    color="Percentage",
    color_continuous_scale="Teal",
)
fig.update_layout(template="plotly_dark")
fig.show()

NameError: name 'time_df' is not defined

### 9f. Feature Correlation

> Now uses Plotly heatmap (consistent with rest of notebook) instead of mixed matplotlib/seaborn.

In [20]:
corr = final_df[[
    "Rating", "Installs", "Sentiment", "Engagement_Score"
]].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    title="📊 Feature Correlation Matrix",
    color_continuous_scale="RdBu_r",
    aspect="equal",
)
fig.update_layout(template="plotly_dark")
fig.show()

NameError: name 'final_df' is not defined

## 10. Activity Trend Analysis

> **Replaces the LSTM** from the original notebook, which was trained on only
> ~20 data points (statistically invalid). A rolling average is more honest
> and interpretable for this data volume.

In [21]:
# Compute daily activity trend
daily_trend = (
    time_df.groupby("Date")
    .size()
    .reset_index(name="Activity")
    .sort_values("Date")
)
daily_trend["Trend"] = daily_trend["Activity"].rolling(window=3, min_periods=1).mean()
daily_trend["Pct_Change"] = daily_trend["Activity"].pct_change().fillna(0) * 100

fig = go.Figure()

fig.add_trace(go.Bar(
    x=daily_trend["Date"].astype(str),
    y=daily_trend["Activity"],
    name="Daily Reviews",
    marker_color="rgba(20, 184, 166, 0.4)",
))
fig.add_trace(go.Scatter(
    x=daily_trend["Date"].astype(str),
    y=daily_trend["Trend"],
    name="3-Day Rolling Avg",
    line=dict(color="#14b8a6", width=3),
))

fig.update_layout(
    title="Review Activity Trend",
    template="plotly_dark",
    xaxis_title="Date",
    yaxis_title="Review Count",
    barmode="overlay",
)
fig.show()

NameError: name 'time_df' is not defined

## 11. Save Outputs

In [22]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

dataset_path = os.path.join(OUTPUT_DIR, "dataset.csv")
model_path   = os.path.join(OUTPUT_DIR, "model.pkl")

final_df.to_csv(dataset_path, index=False)
joblib.dump(model, model_path)

print(f"✅ Dataset saved: {dataset_path} ({final_df.shape[0]} rows, {final_df.shape[1]} cols)")
print(f"✅ Model saved:   {model_path}")

NameError: name 'final_df' is not defined

## 12. Business KPI Summary

In [23]:
print("=" * 50)
print("           BUSINESS INSIGHTS")
print("=" * 50)

top = final_df.sort_values("Engagement_Score", ascending=False)
avg_r = final_df["Rating"].mean()
med_r = final_df["Rating"].median()
avg_s = final_df["Sentiment"].mean()
top_seg = final_df["Segment"].value_counts().idxmax()
cv_mean = cv_scores.mean()
best_hour = hourly.sort_values("Activity", ascending=False)["Hour"].iloc[0]

print(f"  🏆 Top App:         {top['App'].iloc[0]}")
print(f"  ⭐ Avg Rating:      {avg_r:.2f}")
print(f"  ⭐ Median Rating:   {med_r:.2f}")
print(f"  ❤️ Avg Sentiment:   {avg_s:.3f}")
print(f"  ⏰ Peak Hour:       {best_hour}:00")
print(f"  🤖 Top Segment:     {top_seg}")
print(f"  📈 Model CV R²:    {cv_mean:.3f}")
print("=" * 50)

           BUSINESS INSIGHTS


NameError: name 'final_df' is not defined